# Week 4: Evaluation


## Generating Ground Truth Data

In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [7]:
documents = documents_llm

doc = documents[0]
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I just found out about it?', 'Is it too late to start llm-zoomcamp now?', 'If I join late, can I still get a certificate?', 'Do I have to submit the project before submissions close to get the certificate?', 'What’s the deadline for project submission if I want the course certificate?']


In [16]:
from evaluation_utils import llm_structured

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)


['I just found this course late — can I still join and follow along?', 'Is it too late to start the course now, or can new students still sign up?', 'If I join after the course has already started, am I still allowed to participate?', 'Can I still enroll in this course now, even though I discovered it late?', 'Do latecomers still get access, or is it already closed?']


In [18]:
usage.input_tokens, usage.output_tokens

(207, 96)

In [19]:
from evaluation_utils import calc_price

In [20]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000432, 'total_cost': 0.00058725}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still join and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now, or can new students still sign up?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still allowed to participate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still enroll in this course now, even though I discovered it late?',
  'document': '74eb249bbf'},
 {'question': 'Do latecomers still get access, or is it already closed?',
  'document': '74eb249bbf'}]